# Web scraping Falcon 9 launch records
Parses the launch tables from a fixed Wikipedia revision (oldid 1027686922, June 2021) with `requests` and BeautifulSoup.

In [1]:
import sys

import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd

In [2]:
def date_time(table_cells):
    """
    This function returns the data and time from the HTML  table cell
    Input: the  element of a table data cell extracts extra row
    """
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    """
    This function returns the booster version from the HTML  table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out=''.join([booster_version for i,booster_version in enumerate( table_cells.strings) if i%2==0][0:-1])
    return out

def landing_status(table_cells):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out=[i for i in table_cells.strings][0]
    return out


def get_mass(table_cells):
    mass=unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass=mass[0:mass.find("kg")+2]
    else:
        new_mass=0
    return new_mass


def extract_column_from_header(row):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    if (row.br):
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
        
    colunm_name = ' '.join(row.contents)
    
    # Filter the digit and empty names
    if not(colunm_name.strip().isdigit()):
        colunm_name = colunm_name.strip()
        return colunm_name    



def link_text(table_cell):
    """Return the first link's text, or the cell text when the cell has no link."""
    if table_cell.a is not None and table_cell.a.string:
        return table_cell.a.string.strip()
    return table_cell.get_text(" ", strip=True)

In [3]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

In [4]:
# use requests.get() method with the provided static_url
# Wikipedia rejects requests without a descriptive User-Agent.
headers = {"User-Agent": "Falcon9-coursework-scraper/1.0 (portfolio project; fixed revision)"}
response = requests.get(static_url, headers=headers, timeout=30)
response.raise_for_status()

# assign the response to a object 
soup = BeautifulSoup(response.text, 'html.parser')

# Print the page title to verify the BeautifulSoup object was created- Use soup.title attribute
print(soup.title)

<title>List of Falcon 9 and Falcon Heavy launches - Wikipedia</title>


In [5]:
# Use the find_all function in the BeautifulSoup object, with element type `table`
# Assign the result to a list called `html_tables`
html_tables = soup.find_all('table')
# Let's print the third table and check its content
first_launch_table = html_tables[2]
print(str(first_launch_table)[:500])

<table class="wikitable plainrowheaders collapsible" id="mwmQ" style="width: 100%;">
<tbody id="mwmg"><tr id="mwmw"><th id="mwnA" scope="col">Flight No.</th>
<th id="mwnQ" scope="col">Date and<br id="mwng"/>time (<a href="https://en.wikipedia.org/wiki/Coordinated_Universal_Time" id="mwnw" rel="mw:WikiLink" title="Coordinated Universal Time">UTC</a>)</th>
<th id="mwoA" scope="col"><a href="https://en.wikipedia.org/wiki/List_of_Falcon_9_first-stage_boosters" id="mwoQ" rel="mw:WikiLink" title="List


In [6]:
# Step 5: Find all <th> elements (table headers) in the third table
th_elements = first_launch_table.find_all('th')

# Step 6: Initialize an empty list to store column names
column_names = []

# Step 7: Extract and append non-empty column names to the list
for th in th_elements:
    column_name = extract_column_from_header(th)
    if column_name and len(column_name) > 0:
        column_names.append(column_name)

# Step 8: Print the list of column names
print(column_names)

['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


In [7]:
launch_dict= dict.fromkeys(column_names)

# Remove an irrelvant column
del launch_dict['Date and time ( )']

# Let's initial the launch_dict with each value to be an empty list
launch_dict['Flight No.'] = []
launch_dict['Launch site'] = []
launch_dict['Payload'] = []
launch_dict['Payload mass'] = []
launch_dict['Orbit'] = []
launch_dict['Customer'] = []
launch_dict['Launch outcome'] = []
# Added some new columns
launch_dict['Version Booster']=[]
launch_dict['Booster landing']=[]
launch_dict['Date']=[]
launch_dict['Time']=[]

In [8]:
extracted_row = 0
#Extract each table 
for table_number,table in enumerate(soup.find_all('table',"wikitable plainrowheaders collapsible")):
   # get table row 
    for rows in table.find_all("tr"):
        #check to see if first table heading is as number corresponding to launch a number 
        if rows.th:
            if rows.th.string:
                flight_number=rows.th.string.strip()
                flag=flight_number.isdigit()
        else:
            flag=False
        #get table element 
        row=rows.find_all('td')
        #if it is number save cells in a dictonary 
        if flag:
            extracted_row += 1
            # Flight Number value
            launch_dict['Flight No.'].append(flight_number)
            
            datatimelist=date_time(row[0])
            
            # Date value
            date = datatimelist[0].strip(',')
            launch_dict['Date'].append(date)
            
            # Time value
            time = datatimelist[1]
            launch_dict['Time'].append(time)
              
            # Booster version
            bv=booster_version(row[1])
            if not(bv):
                bv=link_text(row[1])
            launch_dict['Version Booster'].append(bv)
            
            # Launch Site
            launch_site = link_text(row[2])
            launch_dict['Launch site'].append(launch_site)
            
            # Payload
            payload = link_text(row[3])
            launch_dict['Payload'].append(payload)
                        
            # Payload Mass
            payload_mass = get_mass(row[4])
            launch_dict['Payload mass'].append(payload_mass)
            
            # Orbit
            orbit = link_text(row[5])
            launch_dict['Orbit'].append(orbit)
            
            # Customer
            customer = link_text(row[6])
            launch_dict['Customer'].append(customer)
            
            # Launch outcome
            launch_outcome = list(row[7].strings)[0]
            launch_dict['Launch outcome'].append(launch_outcome)
            
            # Booster landing
            booster_landing = landing_status(row[8])
            launch_dict['Booster landing'].append(booster_landing)


In [9]:
df= pd.DataFrame({ key:pd.Series(value) for key, value in launch_dict.items() })

In [10]:
from pathlib import Path
Path('../outputs').mkdir(exist_ok=True)
df.to_csv('../outputs/spacex_web_scraped.csv', index=False)
print(df.shape)
df.head()

(121, 11)


,Flight No.,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Version Booster,Booster landing,Date,Time
0,1,CCAFS,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,F9 v1.07B0003.1,Failure,4 June 2010,18:45
1,2,CCAFS,Dragon,0,LEO,NASA,Success,F9 v1.07B0004.1,Failure,8 December 2010,15:43
2,3,CCAFS,Dragon,525 kg,LEO,NASA,Success,F9 v1.07B0005.1,No,22 May 2012,07:44
3,4,CCAFS,SpaceX CRS-1,"4,700 kg",LEO,NASA,Success,F9 v1.07B0006.1,No attempt,8 October 2012,00:35
4,5,CCAFS,SpaceX CRS-2,"4,877 kg",LEO,NASA,Success,F9 v1.07B0007.1,No,1 March 2013,15:10
